# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

In [2]:
# 1. Dictionary to translate the HTML rating words into numbers for filtering
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

In [3]:
def scrape_books(min_rating, max_price):
   
    base_url = "http://books.toscrape.com/catalogue/"
    current_url = "http://books.toscrape.com/catalogue/page-1.html"
    all_books_data = []

    while current_url:
        # Fetch the page content
        
        response = requests.get(current_url)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Identify all book containers on the gallery page
       
        books = soup.find_all('article', class_='product_pod')
        
        for book in books:
            # Extract Rating by looking at the class name 
            
            rating_class = book.find('p', class_='star-rating')['class'][1]
            rating_num = rating_map[rating_class]
            
            # Extract Price and convert to float for comparison
           
            price_text = book.find('p', class_='price_color').text
            price_num = float(re.findall(r'\d+\.\d+', price_text)[0])
            
            # Apply filters: Minimum Rating and Maximum Price
            
            if rating_num >= min_rating and price_num <= max_price:
                
                # Get and clean the link to the individual book page
                
                relative_link = book.find('h3').find('a')['href']
                clean_link = relative_link.replace('../../../', '').replace('../../', '')
                book_url = f"http://books.toscrape.com/catalogue/{clean_link}"
                
                # Visit the specific book page to get UPC and Description
                
                book_res = requests.get(book_url)
                book_soup = BeautifulSoup(book_res.text, 'html.parser')
                
                # Extract UPC using the 'find_next_sibling' method
                
                upc = book_soup.find('th', string='UPC').find_next_sibling('td').text
                
                # Extract Title
                
                title = book_soup.find('h1').text
                
                # Extract Genre from the breadcrumb navigation
                
                genre = book_soup.find('ul', class_='breadcrumb').find_all('li')[2].text.strip()
                
                # Extract Availability
                
                avail = book_soup.find('p', class_='instock availability').text.strip()
                
                # Extract Description (handling cases where it might be missing)
                
                desc_tag = book_soup.find('div', id='product_description')
                description = desc_tag.find_next_sibling('p').text if desc_tag else "No description"
                
                # Store the gathered data in a dictionary
                
                all_books_data.append({
                    "UPC": upc,
                    "Title": title,
                    "Price (£)": price_num,
                    "Rating": rating_num,
                    "Genre": genre,
                    "Availability": avail,
                    "Description": description
                })
        
        #  Find the 'Next' button to move to the next page
        
        next_btn = soup.find('li', class_='next')
        if next_btn:
            next_page_url = next_btn.find('a')['href']
            # Update current_url for the next iteration of the 'while' loop
            if "catalogue/" in next_page_url:
                current_url = f"http://books.toscrape.com/{next_page_url}"
            else:
                current_url = f"http://books.toscrape.com/catalogue/{next_page_url}"
        else:
            current_url = None # End the loop if no 'Next' button is found
            
    # Convert the list of dictionaries into a clean DataFrame
    
    return pd.DataFrame(all_books_data)

In [5]:
# --- EXECUTION ---
# Call the function ( 4+ stars and max £20)

df_results = scrape_books(min_rating=4, max_price=20.0)

In [8]:
# Display the first few rows of scraped data

print(f"Total books found: {len(df_results)}")
df_results.head()

Total books found: 75


,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,ce6396b0f23f6ecc,Set Me Free,17.46,5,Young Adult,In stock (19 available),Aaron Ledbetterâs future had been planned ou...
1,6258a1f6a6dcfe50,The Four Agreements: A Practical Guide to Pers...,17.66,5,Spirituality,In stock (18 available),"In The Four Agreements, don Miguel Ruiz reveal..."
2,6be3beb0793a53e7,Sophie's World,15.94,5,Philosophy,In stock (18 available),A page-turning novel that is also an explorati...
3,657fe5ead67a7767,Untitled Collection: Sabbath Poems 2014,14.27,4,Poetry,In stock (16 available),"More than thirty-five years ago, when the weat..."
4,51653ef291ab7ddc,This One Summer,19.49,4,Sequential Art,In stock (16 available),"Every summer, Rose goes with her mom and dad t..."


In [7]:
# Optional: Save your results to a CSV file

df_results.to_csv("lab_results.csv", index=False)